In [1]:
import dlt
import pandas as pd
import sqlalchemy as sa
from dlt.sources.sql_database import sql_table
from lake import (POSTGRES_URL, DB_TABLES, PIPELINES_DIR,
                  lake_destination, add_metadata, list_lake, read_lake, ensure_bucket)

engine = sa.create_engine(POSTGRES_URL)   # used only to look at / change the database in the demo cells

In [2]:
def build_incremental_resources():
    resources = []
    for name in DB_TABLES:
        r = sql_table(POSTGRES_URL, table=name,
                      incremental=dlt.sources.incremental("updated_at"))
        r.apply_hints(table_name=f"{name}_incremental")
        add_metadata(r, source_system="postgres", source_resource=name, ingestion_type="incremental")
        resources.append(r)
    return resources


def run_incremental(pipeline):
    pipeline.run(build_incremental_resources(), write_disposition="append", loader_file_format="parquet")
    counts = pipeline.last_trace.last_normalize_info.row_counts
    print("New rows loaded:", {k: v for k, v in counts.items() if not k.startswith("_dlt")} or "none")


incr_pipeline = dlt.pipeline(
    pipeline_name="pawhome_incremental",
    destination=lake_destination("silo"),
    dataset_name="postgres",
    pipelines_dir=PIPELINES_DIR,
)

In [3]:
run_incremental(incr_pipeline)
list_lake("postgres/animals_incremental")

New rows loaded: {'adopters_incremental': 2000, 'animals_incremental': 3000, 'applications_incremental': 5000, 'shelters_incremental': 50}


,key,size_kb,last_modified
0,postgres/animals_incremental/ingestion_date=20...,140.0,2026-10-29 20:32:51.760+00:00


In [4]:
run_incremental(incr_pipeline)
list_lake("postgres/animals_incremental")

New rows loaded: none


,key,size_kb,last_modified
0,postgres/animals_incremental/ingestion_date=20...,140.0,2026-10-29 20:32:51.760+00:00


In [5]:
run_incremental(incr_pipeline)
list_lake("postgres/animals_incremental")

New rows loaded: {'animals_incremental': 4}


,key,size_kb,last_modified
0,postgres/animals_incremental/ingestion_date=20...,140.0,2026-10-29 20:32:51.760+00:00
1,postgres/animals_incremental/ingestion_date=20...,5.6,2026-10-29 20:35:55.144+00:00


In [6]:
# The rows that arrived in the LAST run
lake_animals = read_lake("postgres/animals_incremental/")
last_load = lake_animals["_load_id"].astype(float).max()
lake_animals[lake_animals["_load_id"].astype(float) == last_load][["id", "name", "updated_at", "_load_id"]]

,id,name,updated_at,_load_id
3000,3001,Ragnar,2026-10-29 20:34:26.755136,1793306154.2648332
3001,3002,Lake 1,2026-10-29 20:35:00.748982,1793306154.2648332
3002,3003,Lake 2,2026-10-29 20:35:15.877950,1793306154.2648332
3003,3004,Lake 3,2026-10-29 20:35:28.701579,1793306154.2648332


In [7]:
run_incremental(incr_pipeline)
lake_animals = read_lake("postgres/animals_incremental/")

New rows loaded: {'animals_incremental': 1}


In [8]:
# Animals that appear more than once in the lake = animals that were updated
versions = lake_animals[lake_animals.duplicated("id", keep=False)].sort_values(["id", "updated_at"])
versions[["id", "name", "updated_at", "_load_id", "_ingested_at"]]

,id,name,updated_at,_load_id,_ingested_at
3000,3001,Ragnar,2026-10-29 20:34:26.755136,1793306154.2648332,2026-10-29 20:35:54.087130+00:00
3004,3001,Lake 4,2026-10-29 20:36:43.188555,1793306212.4089515,2026-10-29 20:36:52.231573+00:00


In [9]:
latest = (lake_animals.sort_values("updated_at")
                      .drop_duplicates(subset="id", keep="last"))
print("rows in lake:", len(lake_animals), "→ latest versions:", len(latest))
latest[latest.id.isin(versions.id)][["id", "name", "updated_at"]]

rows in lake: 3005 → latest versions: 3004


,id,name,updated_at
3004,3001,Lake 4,2026-10-29 20:36:43.188555


In [11]:
db_ids = set(pd.read_sql("SELECT id FROM animals", engine).id)
in_lake_not_in_db = latest[~latest.id.isin(db_ids)]
print("Deleted in the database but still in the lake:")
in_lake_not_in_db[["id", "name", "updated_at"]]

Deleted in the database but still in the lake:


,id,name,updated_at
3004,3001,Lake 4,2026-10-29 20:36:43.188555


**Conclusion**

| Change in the app | Visible in the lake (append, cursor `updated_at`)? |
|---|---|
| INSERT | ✅ yes – the new rows are added |
| UPDATE | ✅ yes – as a NEW VERSION (the old version stays → history, duplicates by id) |
| DELETE | ❌ no – the deleted row stays in the lake |

Deletes could be captured with soft delete (a `deleted_at` column instead of really deleting),
CDC (reading the database's change log), or by comparing with a periodic full load.
With `id` as the cursor, updates would not be visible either.

In [12]:
ensure_bucket("rustfs")

incr_pipeline_lake2 = dlt.pipeline(
    pipeline_name="pawhome_incremental_rustfs",
    destination=lake_destination("rustfs"),
    dataset_name="postgres",
    pipelines_dir=PIPELINES_DIR,
)
run_incremental(incr_pipeline_lake2)
list_lake("postgres", lake="rustfs")

Bucket pawhome-lake already exists in rustfs
New rows loaded: {'adopters_incremental': 2000, 'animals_incremental': 3003, 'applications_incremental': 5000, 'shelters_incremental': 50}


,key,size_kb,last_modified
0,postgres/adopters_incremental/ingestion_date=2...,192.8,2026-10-29 20:40:06.048+00:00
1,postgres/animals_incremental/ingestion_date=20...,140.2,2026-10-29 20:40:06.085+00:00
2,postgres/applications_incremental/ingestion_da...,262.0,2026-10-29 20:40:06.126+00:00
3,postgres/shelters_incremental/ingestion_date=2...,10.7,2026-10-29 20:40:05.984+00:00


In [13]:
# Run again: nothing new → no new files
run_incremental(incr_pipeline_lake2)

New rows loaded: none
